In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/data/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/etc/src/venv/src-venv/lib/python3.10/site-packages', '/home/fkunneman/IVA_DI/scripts/', '/home/fkunneman/.local/lib/python3.10/site-packages/', '/usr/lib/python3/dist-packages/']


In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
import ucto

import instruction_agent

In [3]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
#tokenizer = AutoTokenizer.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [60]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
#tfidf_file = data_path + 'tfidf/tfidf.model'
#dict_file = data_path + 'tfidf/dict.model'
#tokenizer = ucto.Tokenizer("tokconfig-nld")
#dct = Dictionary.load(dict_file)
#tfidf = TfidfModel.load(tfidf_file)
collection = chroma_client.create_collection(
    name="instruction_db3",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [63]:

instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'

importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,tokenizer,dct,tfidf)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])

In [ ]:
assistant.chat()

Hallo! Ik kan je helpen met 'een reis plannen' of 'een afspraak maken voor een paspoort'. Je kan 'vorige' zeggen als je een stap terug wilt doen, 'volgende' als je naar de volgende stap wilt. Zeg 'herhaal' als je wilt dat ik een stap herhaal. Je kan altijd een vraag stellen. Zeg nu 'reis' of 'paspoort'.


You:  reis


MATCH ['Reis', 'Reis', 'Ik zou graag de reis instructies willen'] , DISTANCE -2.384185791015625e-07 , CAT nav , DO travel CONTEXT b
sel_res ['Reis', 'Reis', 'Ik zou graag de reis instructies willen'] -2.384185791015625e-07 nav travel
Agent: Ik ga je instrueren om een reis met het ov te plannen op 9292.nl. Stap 1: Waar begint je reis? Vul dat in bij 'Van'.


You:  wablief?


MATCH ['Waar is knop?', 'Waar is knop?', "Waar is 'Van'?"] , DISTANCE 0.5559350252151489 , CAT travel , DO Het veld 'Van' staat onder 'Waar wil je heen?' in het midden van het scherm.   CONTEXT 1
sel_res ['Waar is knop?', 'Waar is knop?', "Waar is 'Van'?"] 0.5559350252151489 travel Het veld 'Van' staat onder 'Waar wil je heen?' in het midden van het scherm.  
Agent: Bedoelde je "Waar is knop?"?


You:  nee


MATCH ['Nee', 'Nee', 'Ja'] , DISTANCE 1.1920928955078125e-07 , CAT nav , DO Reject CONTEXT d
sel_res ['Nee', 'Nee', 'Ja'] 1.1920928955078125e-07 nav Reject
Agent: Okay! Zou je het dan nog een keer willen zeggen, in iets andere woorden?


You:  wat bedoelde je?


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MATCH ['Wat bedoel je?', 'Wat bedoel je?', 'Wat zeg je?'] , DISTANCE 0.06645321846008301 , CAT nav , DO clarify CONTEXT 1
sel_res ['Wat bedoel je?', 'Wat bedoel je?', 'Wat zeg je?'] 0.06645321846008301 nav clarify
dynamic system prompt 
                      
                Je bent een spraakassistent die digibeten helpt om een digitale procedure stap voor stap te doorlopen. Dit doe je door instructies te geven die de gebruiker uitvoert. Je instructies gaan over het plannen van een reis met het openbaar vervoer via de website 9292.nl
            De gebruiker probeert de instructies op een laptopscherm uit te voeren en hoeft in reactie op je instructies niet informatie te geven zoals vertrektijd, locatie of persoonlijke gegevens. 
            Maak korte zinnen. Je kan de gebruiker tips geven en helpen met vragen. Hou je uitingen beknopt. Formuleer strikt een reactie op de gebruiker. Formuleer niet uit jezelf een instructie. 
                
                    
De huidige instructie i

In [8]:
query_embedding = assistant.embedding_model.encode(['ik wil naar utrecht'])
#retrieved = assistant.rag.query(query_embeddings=query_embedding,n_results=3,where={'$and': [{'type': {'$in': ['nav',self.domain]}}, {'step_context': {'$in': ['all',self.context]}}]})
print(dir(query_embedding))

['T', '__abs__', '__add__', '__and__', '__array__', '__array_finalize__', '__array_function__', '__array_interface__', '__array_namespace__', '__array_priority__', '__array_struct__', '__array_ufunc__', '__array_wrap__', '__bool__', '__class__', '__class_getitem__', '__complex__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__delitem__', '__dir__', '__divmod__', '__dlpack__', '__dlpack_device__', '__doc__', '__eq__', '__float__', '__floordiv__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__ilshift__', '__imatmul__', '__imod__', '__imul__', '__index__', '__init__', '__init_subclass__', '__int__', '__invert__', '__ior__', '__ipow__', '__irshift__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lshift__', '__lt__', '__matmul__', '__mod__', '__mul__', '__ne__', '__neg__', '__new__', '__or__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdivmod__', '__reduce__',

In [10]:
retrieved = assistant.rag.query(query_texts='ik wil naar utrecht',n_results=3,where={'$and': [{'type': {'$in': ['nav','travel']}}, {'step_context': {'$in': ['all','2']}}]})
print(retrieved)

{'ids': [['bc696322-34c0-4953-a03b-8bc21aeb39a3', 'b40678ca-f23a-4ac7-b270-cc80439ff821', '390fcdcc-7c58-4377-95ca-4372ed21d3ff']], 'embeddings': None, 'documents': [['Ik wil toch een paspoort aanvragen', 'Ik weet het niet', 'Ik wil toch een ov reis plannen']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'action': 'passport', 'type': 'nav', 'step_context': 'all'}, {'step_context': 'all', 'type': 'nav', 'action': 'clarify'}, {'step_context': 'all', 'type': 'nav', 'action': 'travel'}]], 'distances': [[0.39661145210266113, 0.41827982664108276, 0.4186657667160034]]}


In [55]:
import numpy
bow = dct.doc2bow(['ik','wil','naar','utrecht'])
print(bow)
print(len(tfidf.idfs))
print(list(tfidf.idfs.values())[:10])
bow2 = dct.doc2bow(['waar','zit','die','knop','?'])
print(bow2)
dense_vector = numpy.zeros((len(tfidf.idfs), ), float)
# for inner in mx:
#     for index, value in inner:
#         dense_vector[index] = value
# for w in bow:
#     print(len(vector[0]))        
#     vector[0][w[0]] = tfidf.idfs[w[0]]
tfid = tfidf[bow,bow2]
print(tfid)
print(dir(tfid))
print(dir(dense_vector))
for index,value in tfid:
    print(index,value)
    dense_vector[index] = value
#print('DENSE',dense_vector)
print(dense_vector[119],dense_vector[120],dense_vector[722],dense_vector[1820])
print([x for x in dense_vector if x > 0])
#print(tfidf[bow2])
#print(tfidf.id2word(119),tfidf.id2word(722),tfidf.id2word(1820))
#vectors = tfidf[bow,bow2]

#weights = [[x[1] for x in doc] for doc in vectors]
#vector2 = tfidf[bow2]
#print(vectors)
#print(weights)
#print(vector2)
#print(bow)
#loaded_tfidf.idfs(bow)
#vector = loaded_tfidf['ik wil naar utrecht']
print(dir(tfidf))
print(dir(dct))


[(3137, 1)]
2007829
[np.float64(13.388331063935299), np.float64(7.2745168957380075), np.float64(9.70538500927597), np.float64(16.020599279434812), np.float64(11.018025335770986), np.float64(14.42196184181658), np.float64(5.403814162822897), np.float64(7.559412201192293), np.float64(4.930600313382721), np.float64(4.619750331535661)]
[(119, 1), (722, 1), (1820, 1)]
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_adapt_by_suffix', '_load_specials', '_save_specials', '_smart_save', 'add_lifecycle_event', 'chunksize', 'corpus', 'load', 'metadata', 'obj', 'save', 'save_corpus']
['T', '__abs__', '__add__', '__and__', '__array__', '__array_finalize__', '__array_f

ValueError: not enough values to unpack (expected 2, got 1)

In [43]:
x = numpy.zeros((2,5))
for vec in x:
    vec[3] = 0.5
print(x)

[[0.  0.  0.  0.5 0. ]
 [0.  0.  0.  0.5 0. ]]


In [18]:
import ucto
configurationfile = "tokconfig-nld"
tokenizer = ucto.Tokenizer(configurationfile)

tokenizer.process('ik wil naar utrecht.')
tokens = [x.text for x in tokenizer]
print(tokens)

['ik', 'wil', 'naar', 'utrecht', '.']
